# nbdev_tools

> MCP tools for running nbdev commands (prepare, export, test)

In [ ]:
#| default_exp tools.nbdev

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional

from mcp.server.fastmcp import FastMCP

from nbdev_mcp.utils.paths import project_summary, resolve_selector
from nbdev_mcp.utils.subprocess import run, wrap_with_env
from nbdev_mcp.utils.rich import render_result

## nbdev Commands

In [ ]:
#| export
def nbdev_prepare(
    project: Optional[str] = None,
    extra_args: Optional[List[str]] = None,
    use_env: bool = True
) -> Dict[str, Any]:
    """Run nbdev_prepare (export, test, clean notebooks) for the project.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    extra_args : List[str], optional
        Additional arguments to pass to nbdev_prepare.
    use_env : bool
        If True, run in project's conda/mamba environment.
    
    Returns
    -------
    Dict[str, Any]
        Result with command output and status.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    cmd = wrap_with_env(['nbdev_prepare'] + (extra_args or []), p, use_env)
    logs = run(cmd, cwd=p)
    pretty = render_result('nbdev_prepare', project_summary(p), logs)
    return {**logs, 'project': str(p), 'pretty': pretty}

In [ ]:
#| export
def nbdev_export(
    project: Optional[str] = None,
    processors: Optional[List[str]] = None,
    extra_args: Optional[List[str]] = None,
    use_env: bool = True
) -> Dict[str, Any]:
    """Run nbdev_export on the project.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    processors : List[str], optional
        Pre/post processors to apply.
    extra_args : List[str], optional
        Additional arguments to pass to nbdev_export.
    use_env : bool
        If True, run in project's conda/mamba environment.
    
    Returns
    -------
    Dict[str, Any]
        Result with command output and status.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    cmd = ['nbdev_export']
    if processors:
        cmd += ['--procs'] + processors
    if extra_args:
        cmd += extra_args
    
    cmd = wrap_with_env(cmd, p, use_env)
    logs = run(cmd, cwd=p)
    pretty = render_result('nbdev_export', project_summary(p), logs)
    return {**logs, 'project': str(p), 'pretty': pretty}

In [ ]:
#| export
def nbdev_test(
    project: Optional[str] = None,
    path: Optional[str] = None,
    flags: str = '',
    n_workers: Optional[int] = None,
    do_print: bool = False,
    file_re: Optional[str] = None,
    use_env: bool = True
) -> Dict[str, Any]:
    """Run nbdev_test for the project.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    path : str, optional
        Specific path to test.
    flags : str
        Test flags to pass.
    n_workers : int, optional
        Number of parallel workers.
    do_print : bool
        If True, print output during testing.
    file_re : str, optional
        Regex to filter test files.
    use_env : bool
        If True, run in project's conda/mamba environment.
    
    Returns
    -------
    Dict[str, Any]
        Result with command output and status.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    cmd = ['nbdev_test']
    if path:
        cmd += ['--path', path]
    if flags:
        cmd += ['--flags', flags]
    if n_workers is not None:
        cmd += ['--n_workers', str(n_workers)]
    if do_print:
        cmd += ['--do_print']
    if file_re:
        cmd += ['--file_re', file_re]
    
    cmd = wrap_with_env(cmd, p, use_env)
    logs = run(cmd, cwd=p)
    pretty = render_result('nbdev_test', project_summary(p), logs)
    return {**logs, 'project': str(p), 'pretty': pretty}

In [ ]:
#| export
def pytest_run(
    project: Optional[str] = None,
    args: Optional[List[str]] = None,
    use_env: bool = True
) -> Dict[str, Any]:
    """Run pytest on the project's tests/ directory.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    args : List[str], optional
        Arguments to pass to pytest (default ['-q']).
    use_env : bool
        If True, run in project's conda/mamba environment.
    
    Returns
    -------
    Dict[str, Any]
        Result with command output and status.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    cmd = wrap_with_env(['pytest'] + (args or ['-q']), p, use_env)
    logs = run(cmd, cwd=p)
    pretty = render_result('pytest', project_summary(p), logs)
    return {**logs, 'project': str(p), 'pretty': pretty}

## MCP Registration

In [ ]:
#| export
def add_nbdev_tools(mcp: FastMCP) -> None:
    """Attach nbdev build/test tools to the MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    mcp.add_tool(nbdev_prepare)
    mcp.add_tool(nbdev_export)
    mcp.add_tool(nbdev_test)
    mcp.add_tool(pytest_run)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()